In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, it seems to be GROQ API key")
else:
    print("API key found and looks good so far!")

An API key was found, it seems to be GROQ API key


In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "openai/gpt-oss-20b",
    "reasoning_effort": "low",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'openai/gpt-oss-20b',
 'reasoning_effort': 'low',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
## Send the Header and Payload to the API endpoint

response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-48283950-8484-46a8-85d7-ae18cc90df0c',
 'object': 'chat.completion',
 'created': 1793884454,
 'model': 'openai/gpt-oss-20b',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old—and they’re still perfectly edible! The long shelf life comes from honey’s low water content and acidic pH, which create an environment that’s inhospitable to bacteria and microorganisms. So next time you’re craving a sweet snack, you’re basically enjoying a piece of history!',
    'reasoning': 'Need to answer with a fun fact.'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'queue_time': 0.482073383,
  'prompt_tokens': 76,
  'prompt_time': 0.004457909,
  'completion_tokens': 104,
  'completion_time': 0.13323539,
  'total_tokens': 180,
  'total_time': 0.137693299,
  'completion_tokens_details': {'reasoning_tokens': 9}},
 '

## TOO much Cluttery Code 
# We will use Openai Package to make it easier to call the API.



In [4]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI(
api_key=api_key,
base_url="https://api.groq.com/openai/v1"
)

response = openai.chat.completions.create(
    model="openai/gpt-oss-20b", 
    reasoning_effort="low", 
    messages=[
        {"role": "user", "content": "Tell me a fun fact"}
    ]
)

response.choices[0].message.content

'Did you know that octopuses have three hearts? Two pump blood to the gills, while the third pumps it to the rest of the body. And when an octopus swims, the heart that delivers blood to the rest of its body actually stops beating, which is why these clever creatures prefer to crawl rather than swim – it’s less tiring for them!'

## Let's connect with our Local Ollama Model


In [5]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [6]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
    
)

response = ollama.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Tell me a lie"}
    ]
)

print(response.choices[0].message.content)


The Eiffel Tower is actually located in New York City.


In [15]:
response = ollama.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "I have a big presentation tomorrow, and I’m worried I’ll mess it up. What should I do tonight?"}
    ]
)
print(response.choices[0].message.content)

It's normal to feel anxious before a big presentation. Here are some suggestions for tonight to help you prepare and feel more confident:

1. **Review your notes**: Go over your presentation one last time, making sure you're familiar with the material. Focus on key points, visual aids, and any potential questions you might be asked.
2. **Practice your delivery**: Rehearse your presentation in front of a mirror, record yourself, or practice with a friend or family member. This will help you feel more comfortable with the material and confident in your delivery.
3. **Prepare answers to common questions**: Anticipate questions you might be asked and prepare thoughtful responses. This will help you feel more in control and confident.
4. **Get a good night's sleep**: Aim for 7-8 hours of sleep to ensure you're well-rested and alert for your presentation tomorrow.
5. **Eat a nutritious meal**: Fuel your body with a healthy meal to give you energy and focus.
6. **Visualize success**: Take a f

## HOMEWORK EXERCISE ASSIGNMENT
Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

Benefits:

No API charges - open-source
Data doesn't leave your box
Disadvantages:

Significantly less power than Frontier Model

In [7]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [8]:
system_prompt = """
You are a review assistant that analyzes the contents of a website,
and provides a detailed & comprehensive review summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [9]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [10]:
ollama = OpenAI(
    base_url = "http://localhost:11434/v1",
    api_key = "ollama"
)


In [11]:
def message(webData):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + webData}
    ]

In [12]:
def summarize_website(url):
    webData = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=message(webData)
    )
    return response.choices[0].message.content


In [13]:
print(summarize_website("https://colbybryant.com/2018/03/17/review-iron-man-2008/"))

# Website Review Summary

The website is a film review blog that features a review of the 2008 movie "Iron Man" written by Colby R. Bryant. The review is part of a series of reviews for the Marvel Cinematic Universe, and it discusses the film's transformation of the character Tony Stark from a self-indulgent millionaire to a hero who sees the ill-effects of his life's work.

# News and Announcements

There is no news or announcement section on the website. The review is a standalone article, and there are no additional updates or notifications available.

# Review Summary

The review praises Robert Downey Jr.'s casting as Tony Stark and his ability to bring wit and charm to the character. The review notes that the film's origins are adapted to the post-9/11 context, but the core story remains the same. The reviewer praises the film's quick-witted dialogue and action, and notes that it was a runaway hit due to Downey Jr.'s performance.
